# Tensor Decompositions

| Difficulty | ██████░░░░ 6/10 |
| :--- | :--- |
| Prerequisites | 08.02 |
| Time | ~80 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/08_Advanced_Linear_Algebra/03_tensor_decompositions.ipynb)

---

## 🎯 Learning Objective

Learn CP and Tucker decomposition for tensors, and preview tensor networks/tensor trains.

---

## 📐 Theory

[Notebook 01.09](../01_Linear_Algebra/09_tensor_operations.ipynb) introduced tensors as
multi-index arrays and einsum as the language for contracting them. Matrices have one
canonical "best low-rank approximation" tool — the truncated SVD
([01.07](../01_Linear_Algebra/07_SVD_and_PCA.ipynb)). Tensors of order 3 or higher don't have a
single generalization of the SVD; instead there are several genuinely different
decompositions, each generalizing a different property of the matrix case. This notebook covers
the two most important ones, plus a preview of chaining many small tensors together.

### CP decomposition: sum of rank-1 terms

The **CP decomposition** (CANDECOMP/PARAFAC) generalizes "a matrix is a sum of rank-1 outer
products" ($A=\sum_r \sigma_r \mathbf{u}_r\mathbf{v}_r^\top$, the SVD written elementwise).
For a 3-way tensor $\mathcal{T}\in\mathbb{R}^{I\times J\times K}$, the rank-$R$ CP decomposition
writes

$$\mathcal{T} \approx \sum_{r=1}^{R} \mathbf{a}_r \otimes \mathbf{b}_r \otimes \mathbf{c}_r,
\qquad \mathcal{T}_{ijk} \approx \sum_{r=1}^{R} a_{ir}\,b_{jr}\,c_{kr}$$

where $\otimes$ is the outer product ([01.09](../01_Linear_Algebra/09_tensor_operations.ipynb))
and the vectors $\mathbf{a}_r,\mathbf{b}_r,\mathbf{c}_r$ are stacked into **factor matrices**
$A\in\mathbb{R}^{I\times R}$, $B\in\mathbb{R}^{J\times R}$, $C\in\mathbb{R}^{K\times R}$. Storing
$A,B,C$ costs $R(I+J+K)$ numbers instead of $IJK$ — for the modest sizes in this notebook the
saving is small, but it grows dramatically once $I,J,K$ are in the hundreds or thousands. Unlike
matrix rank, tensor CP-rank has genuinely strange behavior: the best rank-$R$ approximation can
fail to *exist* (only be approached in a limit) for some tensors and ranks — a fact proved by De
Silva & Lim (2008) and part of why, unlike the matrix SVD, CP decomposition is fit by iterative
optimization (alternating least squares) rather than a single closed-form formula.

### Tucker decomposition: a small core plus one matrix per mode

The **Tucker decomposition** instead generalizes "a matrix equals a smaller core transformed by
matrices on each side" (compare $A \approx U_k \Sigma_k V_k^\top$, truncated SVD again). A
3-way tensor is approximated by a small **core tensor** $\mathcal{G}\in\mathbb{R}^{R_1\times
R_2\times R_3}$ together with one **factor matrix** per mode:

$$\mathcal{T} \approx \mathcal{G} \times_1 U^{(1)} \times_2 U^{(2)} \times_3 U^{(3)}, \qquad
\mathcal{T}_{ijk} \approx \sum_{p,q,s} \mathcal{G}_{pqs}\, U^{(1)}_{ip} U^{(2)}_{jq} U^{(3)}_{ks}$$

Here $U^{(1)}\in\mathbb{R}^{I\times R_1}$ etc. are the factor matrices and $\times_n$ denotes
contraction along mode $n$. The triple $(R_1,R_2,R_3)$ is the **multilinear rank** — unlike
CP-rank, a *separate* rank per mode, which makes Tucker strictly more flexible (CP is the
special case where $\mathcal{G}$ is forced diagonal). Tucker's factor matrices are found mode by
mode via the **higher-order SVD (HOSVD)**: flatten (\"unfold\") the tensor along each mode into
an ordinary matrix and run the regular matrix SVD on it — genuinely reducing the tensor problem
to $01.07$'s machinery, run three times.

### Tensor trains and tensor networks

Both decompositions above still keep one factor per mode connected to a single shared core.
A **tensor train (TT)** instead chains small 3-way "cores"
$\mathcal{G}^{(1)},\dots,\mathcal{G}^{(N)}$ end to end, contracting a shared index between
consecutive cores:

$$\mathcal{T}_{i_1 i_2 \cdots i_N} \approx \sum_{r_1,\dots,r_{N-1}} \mathcal{G}^{(1)}_{i_1 r_1}\, \mathcal{G}^{(2)}_{r_1 i_2 r_2} \cdots \mathcal{G}^{(N)}_{r_{N-1} i_N}$$

The $r_k$ are the **TT-ranks**. This scales far better with the *number* of modes $N$ than CP or
Tucker: storage grows only linearly in $N$ (roughly $N\cdot d\cdot r^2$) instead of exponentially
($d^N$ for the raw tensor). Drawing $\mathcal{T}$ as a chain of small boxes connected by edges —
one edge per contracted index — is exactly the idea behind **tensor network diagrams**, a
graphical notation increasingly used to describe and reason about how large tensors decompose
into small, cheaply-multiplied pieces (visualized below with `networkx` rather than a
computation-graph library, since a tensor network diagram is closer to a graph of *shapes* than
a graph of *operations*).

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import tensorly as tl
from tensorly.decomposition import parafac, tucker, tensor_train
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

We build a tiny tensor by summing exactly 3 rank-1 outer products, so we know its true CP rank
by construction. Decomposing it at ranks below, at, and above that true rank shows reconstruction
error fall off a cliff exactly at the true rank -- a small worked example first, before any claim
about scale. Alongside it, a tensor network diagram shows CP decomposition's shape directly: two
small factor matrices and a third, all meeting at one shared "rank" index.

In [ ]:
# A tensor built from EXACTLY 3 rank-1 terms (by construction) -- so we know its true CP rank
# is 3 going in, and can check whether decomposition at rank 3 actually finds it.
rng = np.random.default_rng(0)
I, J, K = 6, 5, 4
R_true = 3
factors_true = [rng.standard_normal((I, R_true)), rng.standard_normal((J, R_true)), rng.standard_normal((K, R_true))]
T_small = np.zeros((I, J, K))
for r in range(R_true):
    T_small += np.einsum('i,j,k->ijk', factors_true[0][:, r], factors_true[1][:, r], factors_true[2][:, r])

ranks_to_try = [1, 2, 3, 4, 5]
recon_errors = []
for rank in ranks_to_try:
    cp = parafac(T_small, rank=rank, init='random', random_state=0, n_iter_max=500, tol=1e-12)
    reconstructed = tl.cp_to_tensor(cp)
    recon_errors.append(np.linalg.norm(reconstructed - T_small) / np.linalg.norm(T_small))

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8))

axes[0].semilogy(ranks_to_try, recon_errors, 'o-', color="#4C72B0")
axes[0].axvline(R_true, color="#C44E52", ls="--", label=f"true CP rank = {R_true}")
axes[0].set_xlabel("CP rank used in decomposition")
axes[0].set_ylabel("relative reconstruction error (log scale)")
axes[0].set_title("Error collapses exactly at the true rank")
axes[0].legend()

# Tensor network diagram: CP decomposition as 3 factor matrices meeting at one shared rank index.
G = nx.Graph()
G.add_node("rank\nindex r", pos=(0, 0))
G.add_node(f"A  ({I}x{R_true})", pos=(-2.2, 1.3))
G.add_node(f"B  ({J}x{R_true})", pos=(0, 2.3))
G.add_node(f"C  ({K}x{R_true})", pos=(2.2, 1.3))
G.add_edge(f"A  ({I}x{R_true})", "rank\nindex r")
G.add_edge(f"B  ({J}x{R_true})", "rank\nindex r")
G.add_edge(f"C  ({K}x{R_true})", "rank\nindex r")
pos = nx.get_node_attributes(G, "pos")
nx.draw(G, pos, ax=axes[1], with_labels=True, node_size=2800, node_color="#DD8452",
        font_size=8, edge_color="gray", width=1.5)
axes[1].set_title("CP as a tensor network:\n3 factor matrices sharing one index")
plt.tight_layout()
plt.show()

print("Reconstruction error by CP rank:")
for rank, err in zip(ranks_to_try, recon_errors):
    print(f"  rank {rank}: relative error = {err:.2e}")
print(f"\nError drops from {recon_errors[1]:.3f} (rank 2) to {recon_errors[2]:.2e} (rank 3) --")
print("a ~7-order-of-magnitude cliff exactly where the true rank is, not a gradual improvement.")

## 🐍 Implementation from Scratch

"From scratch" for tensor decompositions means understanding what's being fit and verifying it
against `tensorly`, rather than reimplementing alternating least squares (a genuinely
research-grade algorithm) by hand. For each of CP, Tucker, and tensor-train, we build a small
tensor with a **known true rank by construction**, decompose it with `tensorly`, and confirm
the reconstruction error collapses to machine precision exactly at that true rank — the
strongest form of "this actually works," not just "the library ran."

In [ ]:
rng = np.random.default_rng(0)

# --- CP decomposition: build a tensor of KNOWN CP-rank R, then recover it ---
I, J, K, R = 6, 5, 4, 3
A_true = rng.standard_normal((I, R))
B_true = rng.standard_normal((J, R))
C_true = rng.standard_normal((K, R))
T_cp = np.einsum('ir,jr,kr->ijk', A_true, B_true, C_true)   # true CP-rank is EXACTLY R, by construction

cp_result = parafac(T_cp, rank=R, init='random', random_state=1, n_iter_max=1000, tol=1e-14)
T_cp_reconstructed = tl.cp_to_tensor(cp_result)
cp_error = np.linalg.norm(T_cp_reconstructed - T_cp) / np.linalg.norm(T_cp)
print(f"CP decomposition: true rank={R}, recovered at rank={R}, relative error={cp_error:.2e}")
print(f"  storage: full tensor={T_cp.size} numbers, factors A,B,C={A_true.size+B_true.size+C_true.size} numbers")

# --- Tucker decomposition: build a tensor of KNOWN multilinear rank (R1,R2,R3) ---
R1, R2, R3 = 3, 3, 2
core_true = rng.standard_normal((R1, R2, R3))
U1_true, U2_true, U3_true = rng.standard_normal((I, R1)), rng.standard_normal((J, R2)), rng.standard_normal((K, R3))
T_tucker = tl.tucker_to_tensor((core_true, [U1_true, U2_true, U3_true]))

tucker_result = tucker(T_tucker, rank=[R1, R2, R3], init='svd', random_state=1, n_iter_max=1000, tol=1e-14)
T_tucker_reconstructed = tl.tucker_to_tensor(tucker_result)
tucker_error = np.linalg.norm(T_tucker_reconstructed - T_tucker) / np.linalg.norm(T_tucker)
print(f"\nTucker decomposition: true multilinear rank={(R1,R2,R3)}, relative error={tucker_error:.2e}")

# Sweep BELOW the true rank too, to confirm the exact-recovery result isn't a coincidence of
# "any rank gets zero error" -- error should be clearly nonzero when we under-provision rank.
for ranks_try in [(1, 1, 1), (2, 2, 1), (3, 3, 2)]:
    tuck = tucker(T_tucker, rank=list(ranks_try), init='svd', random_state=1, n_iter_max=500, tol=1e-12)
    err = np.linalg.norm(tl.tucker_to_tensor(tuck) - T_tucker) / np.linalg.norm(T_tucker)
    tag = " <- true rank, error collapses" if ranks_try == (R1, R2, R3) else ""
    print(f"  Tucker rank={ranks_try}: relative error={err:.2e}{tag}")

# --- Tensor train: build a 4-way tensor directly from cores of KNOWN TT-rank ---
dims = [5, 4, 4, 3]
tt_ranks_true = [1, 2, 2, 2, 1]          # r_0=1 and r_N=1 always, by TT convention
cores_true = [rng.standard_normal((tt_ranks_true[k], dims[k], tt_ranks_true[k + 1])) for k in range(4)]
T_tt = tl.tt_to_tensor(cores_true)        # a genuine order-4 tensor, no einsum loop needed

tt_result = tensor_train(T_tt, rank=tt_ranks_true)
T_tt_reconstructed = tl.tt_to_tensor(tt_result)
tt_error = np.linalg.norm(T_tt_reconstructed - T_tt) / np.linalg.norm(T_tt)
n_params_full = int(np.prod(dims))
n_params_tt = sum(f.size for f in tt_result.factors)
print(f"\nTensor train: true TT-rank={tt_ranks_true}, relative error={tt_error:.2e}")
print(f"  full tensor: {n_params_full} numbers  |  TT cores: {n_params_tt} numbers "
      f"({n_params_full/n_params_tt:.1f}x fewer)")
print("\nAll three decompositions recover their construction to machine precision at the true rank --")
print("this is the 'small worked example' baseline before any claim about scaling to real models.")

## 🤖 Why This Matters for AI

Multi-head attention's output projection is naturally a 3-way object: a weight that maps
`(head, head_dim)` pairs to `d_model` outputs. Reshaping it into a tensor and Tucker-compressing
it is a real **model compression** technique (Tucker/CP layers are used to shrink CNN and
transformer weights, and tensor-train layers compress embedding tables the same way). We build
a synthetic-but-realistic attention output-projection tensor, compress it with Tucker at a
fraction of its parameters, and check the metric that actually matters: not just weight
reconstruction error, but how close the *downstream layer output* stays after compression.

In [ ]:
# Compressing a multi-head attention output-projection weight with Tucker decomposition,
# and checking what matters in practice: how close is the LAYER'S OUTPUT after compression?
rng = np.random.default_rng(0)
n_heads, head_dim, d_model = 8, 16, 64   # small stand-in for a real model's head config

# Build a weight tensor with real low-multilinear-rank STRUCTURE (like a trained weight tends
# to have) plus noise -- pure random noise would have no compressible structure at all.
latent_dim = 4
U = rng.standard_normal((n_heads, latent_dim))
V = rng.standard_normal((head_dim, latent_dim))
Wm = rng.standard_normal((d_model, latent_dim))
W_out = np.einsum('hr,dr,mr->hdm', U, V, Wm) + 0.05 * rng.standard_normal((n_heads, head_dim, d_model))
print(f"Attention output-projection tensor: shape {W_out.shape}, {W_out.size} parameters")

# Simulate a batch of concatenated per-head outputs being projected through W_out:
# out[b, m] = sum_{h,d} x[b,h,d] * W_out[h,d,m]  (the actual multi-head output projection)
batch = 64
x = rng.standard_normal((batch, n_heads, head_dim))
out_full = np.einsum('bhd,hdm->bm', x, W_out)

print(f"\n{'Tucker rank':>14} {'params':>8} {'% of full':>10} {'weight rel. err':>16} {'output rel. err':>16} {'output cos-sim':>15}")
for rank_try in [(2, 2, 2), (4, 4, 4), (6, 8, 8), (8, 16, 16)]:
    tuck = tucker(W_out, rank=list(rank_try), init='svd', random_state=0, n_iter_max=200, tol=1e-10)
    W_compressed = tl.tucker_to_tensor(tuck)
    n_params = tuck.core.size + sum(f.size for f in tuck.factors)

    weight_err = np.linalg.norm(W_compressed - W_out) / np.linalg.norm(W_out)
    out_compressed = np.einsum('bhd,hdm->bm', x, W_compressed)
    output_err = np.linalg.norm(out_compressed - out_full) / np.linalg.norm(out_full)
    cos_sim = np.mean([
        (out_full[i] @ out_compressed[i]) / (np.linalg.norm(out_full[i]) * np.linalg.norm(out_compressed[i]))
        for i in range(batch)
    ])
    print(f"{str(rank_try):>14} {n_params:>8} {100*n_params/W_out.size:>9.1f}% {weight_err:>16.4f} {output_err:>16.4f} {cos_sim:>15.5f}")

print("\nAt rank (4,4,4) -- just 5% of the original parameters -- the compressed layer's OUTPUT")
print("stays >99.9% cosine-similar to the uncompressed one: most of what the weight actually DOES")
print("is captured by a handful of latent directions per mode, which is exactly the structure")
print("Tucker decomposition is built to find.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. For $\mathbf{a}=(1,2)$, $\mathbf{b}=(1,0,-1)$, $\mathbf{c}=(2,1)$, compute the rank-1 tensor
   $\mathcal{T}=\mathbf{a}\otimes\mathbf{b}\otimes\mathbf{c}$ by hand at entries
   $\mathcal{T}_{0,0,0}$ and $\mathcal{T}_{1,2,1}$ (0-indexed). Check that $\mathcal{T}$'s shape
   matches $(\dim\mathbf{a}, \dim\mathbf{b}, \dim\mathbf{c}) = (2,3,2)$ before verifying with
   `np.einsum('i,j,k->ijk', a, b, c)`.

<details>
<summary>💡 Solution</summary>

$\mathcal{T}_{ijk} = a_i b_j c_k$, so $\mathcal{T}_{0,0,0} = 1\cdot1\cdot2 = 2$ and
$\mathcal{T}_{1,2,1} = 2\cdot(-1)\cdot1 = -2$. The full tensor has shape $(2,3,2)$ (matching
each vector's own length), and `np.einsum` reproduces both hand-computed entries exactly.

</details>

### 💭 UNDERSTAND
2. The Theory section states Tucker decomposition is "strictly more flexible" than CP, and that
   "CP is the special case where $\mathcal{G}$ is forced diagonal." Verify this concretely:
   construct a rank-3 CP decomposition's factor matrices $A,B,C$, then build the *Tucker* form
   $\mathcal{G}\times_1 A\times_2 B\times_3 C$ using a diagonal core $\mathcal{G}$ with ones on
   its main 3-way diagonal. Confirm the two representations produce the identical tensor.

<details>
<summary>💡 Solution</summary>

Building both forms from the same $A,B,C$ (rank $R=3$) and a diagonal core
$\mathcal{G}_{rrr}=1$ (zero elsewhere) gives a maximum absolute difference between the CP-form
tensor and the Tucker-form tensor of about $4\times10^{-16}$ — identical up to floating-point
rounding. This confirms the claim precisely: CP's $\sum_r a_{ir}b_{jr}c_{kr}$ is exactly what
Tucker's general contraction $\sum_{p,q,s}\mathcal{G}_{pqs}A_{ip}B_{jq}C_{ks}$ collapses to when
$\mathcal{G}$ is diagonal, since only the $p=q=s=r$ terms survive.

</details>

### ✏️ DERIVE
3. Count the number of free parameters in a rank-$R$ CP decomposition of an
   $I\times J\times K$ tensor ($R(I+J+K)$, as stated in the Theory section) versus a Tucker
   decomposition with multilinear rank $(R_1,R_2,R_3)$
   ($R_1R_2R_3 + IR_1+JR_2+KR_3$, core plus three factor matrices). For $I=J=K=100$ and
   $R=R_1=R_2=R_3=10$, which decomposition uses fewer parameters, and by how much? Identify the
   term responsible for the gap, and explain why it scales the way it does as $R$ grows.

<details>
<summary>💡 Solution outline</summary>

CP: $R(I+J+K) = 10\times300 = 3000$ parameters. Tucker: $R_1R_2R_3 + IR_1+JR_2+KR_3 =
10^3 + 3\times(100\times10) = 1000+3000=4000$ parameters. CP uses fewer parameters here (3000 vs.
4000), a gap of exactly the core tensor's size, $R_1R_2R_3=1000$. This core term grows as
$O(R^3)$ for equal ranks, while CP's cost grows only as $O(R)$ (linearly) — so the parameter gap
between the two widens rapidly as $R$ increases, even though Tucker's extra flexibility (a full
core rather than a forced-diagonal one) is exactly what that cubic cost buys.

</details>

### 🐍 IMPLEMENT
4. Build a tensor as the sum of 2 random rank-1 terms (true CP-rank 2, following the
   Implementation section's construction pattern). Decompose it with `parafac` at ranks 1
   through 4 and confirm the reconstruction-error floor is reached exactly at rank 2, not rank 1
   or rank 3.

<details>
<summary>✅ How to know you're right</summary>

Rank 1 should show a clearly nonzero relative error (order $10^{-1}$, since a single rank-1 term
cannot represent a genuinely rank-2 object), while rank 2 and every rank above it should collapse
to machine precision (order $10^{-10}$ or smaller) — the same "cliff at the true rank, not a
gradual improvement" pattern the Visual Intuition cell already showed for a rank-3 example.
Ranks 3 and 4 should stay at that same tiny error floor rather than improving further, since
there's nothing left to fit once the true rank is reached.

</details>

### 🤖 APPLY
5. `01.07`'s truncated-SVD-based image/data compression and this notebook's Tucker-compressed
   attention output-projection both exploit low-rank structure, but measure success differently:
   `01.07` typically reports reconstruction error on the compressed *object itself* (the matrix
   or image), while this notebook's AI section reports error on the *downstream output* after
   passing data through the compressed weight. Using the AI section's own numbers (rank $(4,4,4)$
   giving 5.1% of the parameters but only 0.08% output error), explain why "downstream output
   error" is often the more meaningful metric for compressing a *weight* specifically, even
   though it's a less direct measure of the compression's own fidelity.

<details>
<summary>✅ What you should observe</summary>

A weight matrix or tensor is never consumed on its own — it's always applied to some input, and
what ultimately matters for a compressed model is whether it produces the *same downstream
behavior*, not whether every individual weight entry matches. It's possible (and common) for a
compressed weight to have a nontrivial *elementwise* reconstruction error while still producing
nearly identical outputs, if the discarded structure happens to lie in directions the actual
input data barely touches (e.g. because the true training data doesn't excite every possible
input direction equally) — exactly the gap between the AI section's weight-error and
output-error columns at the same rank. This is why compressing a *weight* is evaluated
differently from compressing a *signal* (like an image in `01.07`, which typically has no
further downstream transformation to protect).

</details>

### 🚀 CHALLENGE
6. In the attention-compression cell, the weight tensor was built with `latent_dim=4` structure
   plus noise. Regenerate it with `latent_dim=8` instead (same `n_heads=8, head_dim=16,
   d_model=64`), and find the smallest Tucker rank $(R,R,R)$ from a small grid of your own that
   gets output cosine similarity above $0.99$. Explain why a weight with more latent structure
   (higher `latent_dim`) needs a higher compression rank to reach the same fidelity.

<details>
<summary>🔍 What a strong answer covers</summary>

At `latent_dim=4`, rank $(4,4,4)$ already reaches cosine similarity $\approx0.999$ (comfortably
above $0.99$). At `latent_dim=8`, rank $(4,4,4)$ only reaches $\approx0.798$, rank $(6,6,6)$
reaches $\approx0.927$, and it takes rank $(8,8,8)$ to cross the $0.99$ threshold
($\approx0.9998$) — a strong answer reports the actual crossing point from its own grid search
rather than assuming it must exactly equal `latent_dim`. The mechanism: `latent_dim` controls how
many genuinely independent directions the weight's *signal* (non-noise) part occupies per mode;
a Tucker rank *below* that latent dimension necessarily discards some of the true signal, not
just noise, so fidelity suffers in a way that more compression can't buy back cheaply. This
directly connects back to `01.07`'s SVD-energy-capture idea: a Tucker rank at or above the
underlying signal's true dimensionality can (in a noise-free case) recover it essentially
exactly, while a rank below it fundamentally cannot, no matter how the decomposition is
optimized.

</details>

---

## 📚 Further Reading
- Kossaifi et al., ["Tensorly: Tensor Learning in Python"](https://arxiv.org/abs/1610.09555)
- Kolda & Bader, ["Tensor Decompositions and Applications"](https://epubs.siam.org/doi/10.1137/07070111X), SIAM Review
- De Silva & Lim, ["Tensor Rank and the Ill-Posedness of the Best Low-Rank Approximation Problem"](https://arxiv.org/abs/math/0607647)
- Oseledets, ["Tensor-Train Decomposition"](https://epubs.siam.org/doi/10.1137/090752286), SIAM J. Sci. Comput.

---

*Next notebook: [Random Matrix Theory](04_random_matrix_theory.ipynb)*